## Imports

In [30]:
import pandas as pd
import numpy as np
import scipy
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

### Leitura do CSV baixado do Kaggle

In [31]:
caminho_arquivo = "AI_Impact_Student_Life_2026.csv"

df_completo = pd.read_csv(
    caminho_arquivo,
    sep=",",
    encoding="utf-8",
    decimal="."
)

df_completo.columns = df_completo.columns.str.strip()

### Seleção apenas das colunas de interesse

In [32]:
colunas_interesse = [
    "Student_ID",
    "Task_Frequency_Daily",
    "Main_Usage_Case",
    "GPA_Baseline",
    "GPA_Post_AI"
]

df = df_completo[colunas_interesse].copy()

### Inspeção inicial

In [ ]:
print("Formato do DataFrame (linhas, colunas):", df.shape)
print("\nPrimeiras linhas:")
print(df.head())

print("\nTipos de dados por coluna:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nResumo estatístico (colunas numéricas):")
print(df.describe())

# ---- Identificação de dados duplicados ----

# Duplicatas exatas (todas as colunas iguais)
qtd_duplicadas_exatas = df.duplicated().sum()
print(f"\nQuantidade de linhas totalmente duplicadas: {qtd_duplicadas_exatas}")

# Duplicatas por Student_ID (mesmo aluno aparecendo mais de uma vez)
qtd_ids_duplicados = df.duplicated(subset="Student_ID").sum()
print(f"Quantidade de Student_ID duplicados: {qtd_ids_duplicados}")

if qtd_ids_duplicados > 0:
    print("\nExemplos de registros com Student_ID duplicado:")
    print(df[df.duplicated(subset="Student_ID", keep=False)].sort_values(by="Student_ID").head(10))

# ---- Tratamento: remoção das duplicatas por Student_ID ----
df = df.drop_duplicates(subset="Student_ID", keep="first")
print(f"\nFormato do DataFrame após remoção de duplicatas por Student_ID: {df.shape}")

### Ajustes de formatação

In [35]:
# Garante que as colunas de GPA são numéricas
df["GPA_Baseline"] = pd.to_numeric(df["GPA_Baseline"], errors="coerce")
df["GPA_Post_AI"] = pd.to_numeric(df["GPA_Post_AI"], errors="coerce")

# Garante que a frequência de uso também é numérica
df["Task_Frequency_Daily"] = pd.to_numeric(df["Task_Frequency_Daily"], errors="coerce")

# Remove linhas com dados faltando nas colunas essenciais
df = df.dropna(subset=["GPA_Baseline", "GPA_Post_AI"])

# Remove linhas totalmente vazias
df = df.dropna(how="all")

### Coluna calculada: variação de GPA

In [36]:
df["GPA_Variacao"] = df["GPA_Post_AI"] - df["GPA_Baseline"]

### Correlação de Pearson: Task_Frequency_Daily (Frequência diária do uso de agentes IA) x GPA_Variacao

In [ ]:
coef_pearson, p_valor_pearson = stats.pearsonr(df["Task_Frequency_Daily"], df["GPA_Variacao"])

print(f"\nCoeficiente de correlação de Pearson: {coef_pearson:.4f}")
print(f"P-valor (Pearson): {p_valor_pearson:.5f}")

### Correlação de Spearman: Task_Frequency_Daily (Frequência diária do uso de agentes IA) x GPA_Variacao

In [ ]:
coef_spearman, p_valor_spearman = stats.spearmanr(df["Task_Frequency_Daily"], df["GPA_Variacao"])

print(f"\nCoeficiente de correlação de Spearman: {coef_spearman:.4f}")
print(f"P-valor (Spearman): {p_valor_spearman:.5f}")

### Interpretação da força da correlação

In [ ]:
def interpretar_correlacao(r):
    r_abs = abs(r)
    if r_abs < 0.1:
        return "praticamente nenhuma correlação"
    elif r_abs < 0.3:
        return "correlação fraca"
    elif r_abs < 0.5:
        return "correlação moderada"
    elif r_abs < 0.7:
        return "correlação forte"
    else:
        return "correlação muito forte"

def resumo_correlacao(nome, r, p):
    direcao = "positiva" if r > 0 else "negativa"
    forca = interpretar_correlacao(r)
    significativo = "estatisticamente significativa" if p < 0.05 else "não estatisticamente significativa (p >= 0.05)"
    print(f"\n[{nome}] correlação {forca} e {direcao} (r = {r:.4f}). Resultado {significativo} (p = {p:.5f}).")

resumo_correlacao("Pearson", coef_pearson, p_valor_pearson)
resumo_correlacao("Spearman", coef_spearman, p_valor_spearman)

### Calculando Quartis da variação de GPA

In [ ]:
gpa_var_ordenada = df["GPA_Variacao"].sort_values().reset_index(drop=True)

q1 = gpa_var_ordenada.quantile(0.25)
mediana = gpa_var_ordenada.median()
q3 = gpa_var_ordenada.quantile(0.75)
iqr = q3 - q1

print(f"Quantidade de observações: {len(gpa_var_ordenada)}")
print(f"1º quartil (Q1): {q1:.2f}")
print(f"Mediana (Q2): {mediana:.2f}")
print(f"3º quartil (Q3): {q3:.2f}")
print(f"Intervalo interquartil (IQR): {iqr:.2f}")
print(f"Valor mínimo: {gpa_var_ordenada.min():.2f} | Valor máximo: {gpa_var_ordenada.max():.2f}")

### Boxplot da variação de GPA

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(x=df["GPA_Variacao"], ax=ax, color="#9ecae1", width=0.45)

for valor, rotulo in [(q1, "Q1"), (mediana, "Mediana"), (q3, "Q3")]:
    ax.annotate(f"{rotulo} = {valor:.2f}", xy=(valor, 0.225), xytext=(valor, 0.33),
                ha="center", fontsize=10,
                arrowprops=dict(arrowstyle="-", color="gray"))

ax.set_title("Boxplot da variação de GPA (GPA_Post_AI − GPA_Baseline)")
ax.set_xlabel("Variação de GPA")
plt.tight_layout()
plt.show()

### Assimetria (skewness) das variáveis principais

In [ ]:
coluna = "GPA_Variacao"
dados = df[coluna].dropna()
coef_assimetria = stats.skew(dados)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(dados, kde=True, ax=ax, color="#4a90d9", bins=30)
ax.axvline(dados.mean(), color="red", linestyle="--", linewidth=1.2, label=f"Média = {dados.mean():.2f}")
ax.axvline(dados.median(), color="green", linestyle="--", linewidth=1.2, label=f"Mediana = {dados.median():.2f}")

ax.set_title(f"Variação de GPA (Pós - Baseline)\nAssimetria (skew) = {coef_assimetria:.4f}", fontsize=12)
ax.set_xlabel(coluna)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Assimetria de {coluna}: {coef_assimetria:.4f}")

# **Resultados da Análise Exploratória**

### **Hipóteses refutadas**

Ao final de nossa análise do dataset vindo do artigo do Kaggle, notamos que algumas de nossas hipóteses estavam **incorretas**. Sendo estas as seguintes:

- "Supõe-se que o acesso à IA provocou uma menor confiabilidade entre notas dos alunos e grau de aprendizado, pois talvez o uso exacerbado e antiético de agentes de IA possa se refletir em ocilações bruscas de nota frequentes para os estudantes."

- "Supõe-se que o acesso à agentes de IA reduziu as taxas de aprendizado dos universitários, pois, se for notado incosistências na distribuição dos índices de rendimento após o uso de agentes de IA, isso pode indicar uma insegurança no grau de aprendizado dos universitários."

As provas estatísticas para tal refutação partem do resultado dos **coeficientes de Correlação de Pearson e Spearman**, dado que ambos ficaram em **valores próximos e bem abaixo de 0.1**, indicando que os índices de rendimento gerais dos universitários (GPA) não tiveram alterações influenciadas pela quantidade de usos diários de agentes de inteligência artificial.

### **Hipóteses confirmadas**

Notou-se também que, apesar dessa falta de correlação estatística, baseado nos dados do boxplot e gráfico de simetria, houve um aumento nos GPA's após a implementação de agentes de IA na vida estudantil. O que confirmou uma das nossas hipóteses:

- "Supõe-se que, Após o acesso à IA, houve um aumento de notas nas universidades, pois o uso de IA's generativa para criação de materiais de estudo organizados ou o seu uso para auxiliar na execução de projetos da faculdade podem ter sido fatores que influenciaram as notas de forma a aumentar o índice de rendimento geral dos universitários."